# Baseline eval of the unpruned teacher: HumanEval-TS

Part of plan.md step 1.2's baseline eval suite (`Qwen3-30B-A3B-Base` as-is --
the number every pruned candidate in Phase 2/4 gets compared to), covering
just the **primary, gates-every-decision** MultiPL-E HumanEval-TS task
(`multiple-ts`), pass@1 at temperature 0.2 / nucleus top-p 0.95 --
`bigcode-evaluation-harness`'s own defaults already match the papers'
settings exactly. Run independently from the rest of the suite (MBPP-TS,
MMLU, ARC-Challenge, Winogrande -- each gets its own notebook) so a failure
or a long run in one doesn't block or hide the others.

`tools/bigcode_eval_main.py` (a vendored copy of the harness's own main.py --
`pip install`ing bigcode-eval does not ship it) calls `bonsai.eval.register()`
itself before parsing `--tasks`, since `accelerate launch` runs as its own
subprocess and never sees this notebook kernel's state.

**`batch_size=8`, not the harness's namesake-suggesting-bigger default:**
the first attempt at `batch_size=32` sat for over an hour without finishing.
Batched generation waits for the slowest sequence in each batch to hit a stop
token or `max_length_generation`; with 32 sequences per batch the odds that
at least one doesn't stop early (dragging the whole batch to the cap) are
much higher than with a smaller batch, and this MoE model's decode cost
doesn't amortize across a batch as cleanly as a dense model's would -- 32
different sequences can route to 32 different sets of experts at every layer,
so the per-step memory-bandwidth cost doesn't shrink per-sequence the way
batching usually buys you. `batch_size=8` (and `max_length_generation=512`,
the harness's own default, down from an initial 1024) finished this task in
about 24 minutes.

**Result: pass@1 = 0.6708 (67.1%).**

Needs a rented GPU (1x 80GB, per plan.md's sizing table) with the `eval`
extra installed: `pip install -e ".[eval]"`, plus Node.js + `typescript`
(`tsc`) for MultiPL-E's TypeScript execution -- neither ships with the
Python env `setup_instance.sh` builds.

In [1]:
import json

from bonsai.config import load_config
from bonsai.tracking import new_run, save_config_snapshot

config = load_config("../configs/qwen3-30b-a3b.toml")
run_dir = new_run("baseline-eval")
save_config_snapshot(run_dir, config)

print(f"model: {config.model.repo_id}")
print(f"run dir: {run_dir}")

model: Qwen/Qwen3-30B-A3B-Base
run dir: runs/20260915T035727Z-4b4e40-baseline-eval


## Primary: HumanEval-TS and MBPP-TS

One `accelerate launch` per task rather than a combined `--tasks multiple-ts,multiple-mbpp-ts`
call, so a failure in one doesn't take the other's results down with it, and
each gets its own `--metric_output_path` / `--save_generations_path`. Shell
cells stream `accelerate`'s progress bars and per-problem execution output
straight into the notebook.

In [2]:
import os

os.environ["MODEL_REPO_ID"] = config.model.repo_id
os.environ["MODEL_REVISION"] = config.model.revision
os.environ["RUN_DIR"] = str(run_dir)

In [3]:
!stdbuf -o0 -e0 accelerate launch ../tools/bigcode_eval_main.py \
    --model "$MODEL_REPO_ID" --revision "$MODEL_REVISION" --precision bf16 \
    --tasks multiple-ts \
    --max_length_generation 512 --batch_size 8 \
    --allow_code_execution --save_generations \
    --metric_output_path "$RUN_DIR/humaneval-ts.json" \
    --save_generations_path "$RUN_DIR/humaneval-ts-generations.json" \
    2>&1 | tee /tmp/live_humaneval_ts.log

assert _exit_code == 0, "accelerate launch failed for multiple-ts"  # noqa: F821 -- set by the ! shell magic above at runtime

The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_processes` was set to a value of `1`
	`--num_machines` was set to a value of `1`
	`--mixed_precision` was set to a value of `'no'`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.


Selected Tasks: ['multiple-ts']
Loading model in bf16
Loading weights:   0%|          | 0/531 [00:00<?, ?it/s]

Loading weights:   1%|          | 4/531 [00:00<00:33, 15.56it/s]

Loading weights:   1%|          | 6/531 [00:01<01:52,  4.67it/s]

Loading weights:   3%|▎         | 15/531 [00:01<00:38, 13.23it/s]

Loading weights:   3%|▎         | 18/531 [00:02<01:02,  8.22it/s]

... (409 outputs left out)

 99%|█████████▊| 157/159 [01:15<00:00,  2.49it/s]

 99%|█████████▉| 158/159 [01:16<00:00,  2.69it/s]

100%|██████████| 159/159 [01:16<00:00,  2.08it/s]


{
  "multiple-ts": {
    "pass@1": 0.6708074534161491
  },
  "config": {
    "prefix": "",
    "do_sample": true,
    "temperature": 0.2,
    "top_k": 0,
    "top_p": 0.95,
    "n_samples": 1,
    "eos": "<|endoftext|>",
    "seed": 0,
    "model": "Qwen/Qwen3-30B-A3B-Base",
    "modeltype": "causal",
    "peft_model": null,
    "revision": "main",
    "use_auth_token": false,
    "trust_remote_code": false,
    "tasks": "multiple-ts",
    "instruction_tokens": null,
    "batch_size": 8,
    "max_length_generation": 512,
    "precision": "bf16",
    "load_in_8bit": false,
    "load_in_4bit": false,
    "left_padding": false,
    "limit": null,
    "limit_start": 0,
    "save_every_k_tasks": -1,
    "postprocess": true,
    "allow_code_execution": true,
    "generation_only": false,
    "load_generations_path": null,
    "load_data_path": null,
    "metric_output_path": "runs/20260915T035727Z-4b4e40-baseline-eval/humaneval-ts.json",
    "save_generations": true,
    "load_generations_in

In [4]:
humaneval_ts = json.loads((run_dir / "humaneval-ts.json").read_text())
print("humaneval-ts:", humaneval_ts)

humaneval-ts: {'multiple-ts': {'pass@1': 0.6708074534161491}, 'config': {'prefix': '', 'do_sample': True, 'temperature': 0.2, 'top_k': 0, 'top_p': 0.95, 'n_samples': 1, 'eos': '<|endoftext|>', 'seed': 0, 'model': 'Qwen/Qwen3-30B-A3B-Base', 'modeltype': 'causal', 'peft_model': None, 'revision': 'main', 'use_auth_token': False, 'trust_remote_code': False, 'tasks': 'multiple-ts', 'instruction_tokens': None, 'batch_size': 8, 'max_length_generation': 512, 'precision': 'bf16', 'load_in_8bit': False, 'load_in_4bit': False, 'left_padding': False, 'limit': None, 'limit_start': 0, 'save_every_k_tasks': -1, 'postprocess': True, 'allow_code_execution': True, 'generation_only': False, 'load_generations_path': None, 'load_data_path': None, 'metric_output_path': 'runs/20260915T035727Z-4b4e40-baseline-eval/humaneval-ts.json', 'save_generations': True, 'load_generations_intermediate_paths': None, 'save_generations_path': 'runs/20260915T035727Z-4b4e40-baseline-eval/humaneval-ts-generations.json', 'save_